
<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 15px; height: 80px">

# Random Forests and ExtraTrees


_Authors:_ Tim Book (DC), Katie Sylvia (NH), et al

---

## Random Forests
Random forests are a tree-based ensemble method similar to bagging a decision tree, with the key difference that only a subset of the full $p$ variables are considered at each split.

For a problem with $p$ features, it is traditional to use:

- $\sqrt{p}$ (rounded down) features in each split for a classification problem.
- $p/3$ (rounded down) with a minimum node size of 5 as the default for a regression problem.

While this is a guideline suggested by the origial inventor (Breiman, 2001), Hastie and Tibshirani (authors of ESL and ISLR) have suggested treating the number of parameters to use at each split as a tuning parameter.

In [73]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

## Scenario 1: Titanic Data (again)

In [3]:
titanic = pd.read_csv('../datasets/titanic_clean.csv')
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


In [52]:
# Quickly make X, y
X = pd.get_dummies(
    columns=['Pclass', 'Sex', 'Embarked'],
    data=titanic.drop(columns=['PassengerId', 'Name', 'Survived']),
    drop_first=True
)

y = titanic['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.6, random_state=42
)

In [53]:
1 - y_train.mean()

0.5971896955503513

In [54]:
rf = RandomForestClassifier(oob_score=True, max_features='sqrt')
rf.fit(X_train, y_train)

RandomForestClassifier(max_features='sqrt', oob_score=True)

In [55]:
rf.score(X_train, y_train)

0.9953161592505855

In [56]:
rf.score(X_test, y_test)

0.7649122807017544

In [57]:
rf.oob_score_

0.7939110070257611

In [58]:
pd.DataFrame({
    'variable': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

,variable,importance
3,Fare,0.269569
0,Age,0.260783
6,Sex_male,0.253575
5,Pclass_3,0.078632
1,SibSp,0.046179
2,Parch,0.034468
4,Pclass_2,0.030123
8,Embarked_S,0.022534
7,Embarked_Q,0.004139


## Scenario 2: Spam!
Used throughout ESL. Data dictionary [here](https://archive.ics.uci.edu/dataset/94/spambase)!

In [59]:
names_raw = open('../datasets/spam-colnames.txt', 'r').readlines()
names = [line.split(':')[0] for line in names_raw[33:]] + ['is_spam']
spam = pd.read_csv('../datasets/spam.csv', sep='\s+', names=names)
spam.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


In [60]:
X = spam.drop(columns=['is_spam'])
y = spam['is_spam']

In [69]:
1 - y.mean()

0.6059552271245381

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1234)

In [62]:
X.shape

(4601, 57)

In [64]:
# YOU! Create me a benchmark RF using any parameter values you like
rf = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    max_depth=None,
    oob_score=True,
    random_state=2023
)

In [65]:
rf.fit(X_train, y_train)

RandomForestClassifier(max_features='sqrt', n_estimators=500, oob_score=True,
                       random_state=2023)

In [70]:
rf.score(X_train, y_train)

1.0

In [71]:
rf.score(X_test, y_test)

0.947002606429192

In [72]:
rf.oob_score_

0.9533333333333334

### Search the parameter space!... for Spam!

In [74]:
p = X.shape[1]

In [76]:
params = {
    'max_features': np.arange(1, p + 1),
    'max_depth': np.append(np.arange(1, 9), None),
    'min_samples_leaf': np.arange(1, 31)
}

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=2023
)

rs = RandomizedSearchCV(
    rf, param_distributions=params, n_iter=100, cv=5, n_jobs=8 
)

In [77]:
%%time
rs.fit(X_train, y_train)

CPU times: user 3.25 s, sys: 181 ms, total: 3.43 s
Wall time: 44.7 s


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=2023),
                   n_iter=100, n_jobs=8,
                   param_distributions={'max_depth': array([1, 2, 3, 4, 5, 6, 7, 8, None], dtype=object),
                                        'max_features': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
       52, 53, 54, 55, 56, 57]),
                                        'min_samples_leaf': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])})

In [78]:
rs.score(X_train, y_train)

0.9634782608695652

In [79]:
rs.score(X_test, y_test)

0.9374456993918332

In [81]:
rs.best_params_

{'min_samples_leaf': 1, 'max_features': 25, 'max_depth': 8}

### Can also try: Extra Trees!

In [82]:
et = ExtraTreesClassifier(n_estimators=500)
et.fit(X_train, y_train)

ExtraTreesClassifier(n_estimators=500)

In [83]:
et.score(X_train, y_train)

1.0

In [84]:
et.score(X_test, y_test)

0.94874022589053

# Conclusions
The random forest model is probably the most powerful model we've seen so far. It combines the simplicity of a decision tree with the brute-force of ensemble techniques. It's a popular choice for both classification and regression, and is an excellent exercise in parameter searching as well. 